# Curated spike-train analysis
Loads only curated Sorting and QC duration metadata. No raw Recording/Sorting, preprocessing or sorter call. The configured window, short-recording policy and normalization remain methodological choices, not established consensus.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from utils import load_config, read_json
config = load_config(ROOT / 'config.yaml')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spikeinterface as si
from decimal import Decimal, ROUND_CEILING
qc_folder = Path(config['paths']['qc'])
sorting_curated = si.load(qc_folder / 'sorting_curated')
source = read_json(qc_folder / 'metadata/analysis_source.json')
parameters = config['isi_analysis']
print('Conventions:', parameters)
fs, duration = source['sampling_frequency_hz'], source['duration_sec']
start = Decimal(str(parameters['interval_start_sec']))
end = start+Decimal(str(parameters['interval_sec']))
first = int((start*Decimal(str(fs))).to_integral_value(rounding=ROUND_CEILING))
last = min(source['num_samples'], int((end*Decimal(str(fs))).to_integral_value(rounding=ROUND_CEILING)))
used_sec = max(0., min(parameters['interval_sec'], duration-float(start)))
complete = duration-float(start) >= parameters['interval_sec']
print('Window seconds used:', used_sec, '; complete:', complete)
if not parameters['conventions_acknowledged']:
    raise ValueError('Acknowledge conventions in config before interpreting this analysis.')
skip = not complete and parameters['short_recording_policy'] == 'skip'


In [ ]:
rows = []
for unit in sorting_curated.unit_ids:
    frames = sorting_curated.get_unit_spike_train(unit)
    selected = frames[(frames >= first) & (frames < last)]
    isi_ms = np.diff(selected)/fs*1000
    rows.append(dict(unit_id=unit, spike_count=len(selected),
        firing_rate_hz=np.nan if skip or used_sec <= 0 else len(selected)/used_sec,
        median_all_isi_ms=np.nan if skip or not len(isi_ms) else np.median(isi_ms),
        complete_interval=complete, status='short_recording_skipped' if skip else 'exploratory'))
display(pd.DataFrame(rows, columns=['unit_id','spike_count','firing_rate_hz','median_all_isi_ms','complete_interval','status']))


In [ ]:
if not skip:
    for unit in sorting_curated.unit_ids[:3]:
        frames = sorting_curated.get_unit_spike_train(unit)
        selected = frames[(frames >= first) & (frames < last)]
        isi_ms = np.diff(selected)/fs*1000
        retained = isi_ms[(isi_ms >= parameters['isi_min_ms']) & (isi_ms <= parameters['isi_max_ms'])]
        if len(retained):
            weights = None if parameters['histogram_normalization'] == 'density' else np.ones(len(retained))/len(retained)
            plt.hist(retained, bins=np.linspace(parameters['isi_min_ms'], parameters['isi_max_ms'], 31),
                weights=weights, density=parameters['histogram_normalization'] == 'density', alpha=.4, label=str(unit))
    plt.xlabel('ISI (ms)'); plt.ylabel(parameters['histogram_normalization'])
    if len(sorting_curated.unit_ids):
        plt.legend(title='Candidate unit')
else:
    print('ISI exploration skipped by the configured short-recording policy.')
# Exploratory fixed bins above differ from the paper median/15 fit in 04_analysis.py.


In [ ]:
result_file = Path(config['paths']['analysis']) / 'metrics/analysis.csv'
if result_file.exists():
    display(pd.read_csv(result_file))
# Extension point: separately document any future biological/network analysis.
